In [1]:
from huggingface_hub import notebook_login
print("Log in using your personal WRITE token:")
notebook_login()

Log in using your personal WRITE token:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
# Upgrade torchao to the version required by PEFT
!pip install -U torchao

# Force upgrade peft and transformers just to keep everything perfectly in sync
!pip install -U peft transformers trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found

In [3]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 2. Define the model addresses
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
HF_ADAPTER_ID = "rose00009/code-reviewer"

print("Step 1: Loading base model configurations")
# Load the base model layers in native bfloat16 memory optimization
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print("Step 2: Fusing your custom adapter layers over the base model")
model = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)

print("Step 3: Initializing the high-level text generation pipeline")
# Wrap the combined neural network assembly into an optimized pipeline object
code_review_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Pipeline is online and fully configured")

Step 1: Loading base model configurations


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Step 2: Fusing your custom adapter layers over the base model


Step 3: Initializing the high-level text generation pipeline
Pipeline is online and fully configured


In [4]:
test_code = """
def calculation(items):
    out = []
    for x in items:
        out.append(x * 2)
    return out
"""

messages = [
    {"role": "system", "content": "You are an expert code reviewer."},
    {"role": "user", "content": f"Review this Python function and offer optimization recommendations:\n{test_code}"}
]

print("Generating code review evaluation...")

outputs = code_review_pipeline(
    messages,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.4,
    top_p=0.9
)

response_content = outputs[0]["generated_text"][-1]["content"]

print("\n--- Code Review Output ---")
print(response_content)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating code review evaluation...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- Code Review Output ---
Issue Identification: Performance Issue - Inefficient List Creation

Problem Analysis: The function uses a list comprehension, which is the most efficient way to create a new list by applying an operation to each item of an existing iterable. However, the code manually appends each result to a new list using `out.append()`. This is less readable than the list comprehension syntax and can be slightly less performant due to the overhead of the `append` method calls.

Solution:
```python
from typing import List, Any

def calculation(items: List[Any]) -> List[Any]:
    return [x * 2 for x in items]
```
